# PostgreSQL 数据管道与数据清洗

运行第一个单元格时会弹出输入框，请输入数据库密码。密码不会被写入代码，也不会保存在 notebook 的输出中。

In [1]:
import getpass
import psycopg2
import pandas as pd

# 数据库连接参数：以下为 PostgreSQL 安装时的默认值，如与你的设置不同请自行修改
DB_HOST = "localhost"
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PORT = "5432"


# --- 步骤 1: 建立连接（密码在运行时输入）---
def connect_to_db():
    password = getpass.getpass("请输入数据库密码：")
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            dbname=DB_NAME,
            user=DB_USER,
            password=password,
            port=DB_PORT,
            options="-c search_path=public",
        )
        print("数据库连接成功！")
        return conn
    except psycopg2.Error:
        print("数据库连接失败，请检查数据库是否已启动，以及密码和连接参数是否正确。")
        return None


# --- 步骤 2: 跨表提取数据 ---
def extract_main_dataset(conn):
    query = """
    SELECT
        u.age,
        r.region_name,
        p.category,
        t.amount,
        t.transaction_date
    FROM transactions t
    LEFT JOIN users u ON t.user_id = u.user_id
    LEFT JOIN products p ON t.product_id = p.product_id
    LEFT JOIN regions r ON u.region_id = r.region_id;
    """
    try:
        with conn.cursor() as cur:
            cur.execute(query)
            columns = [col[0] for col in cur.description]
            df = pd.DataFrame(cur.fetchall(), columns=columns)
        print("数据提取成功！")
        print(df)
        return df
    except psycopg2.Error:
        print("数据提取失败，请检查表是否已创建并已导入数据。")
        return None


# 连接在最后一个单元格关闭
df = None
conn = connect_to_db()
if conn:
    df = extract_main_dataset(conn)


数据库连接成功！
数据提取成功！
      age region_name category  amount transaction_date
0    53.0          华北     生鲜食品    8841       2024-09-14
1    31.0          华北     日用百货    1412       2025-04-26
2    45.0          华东     电子产品   16757       2025-08-05
3    35.0          华南     日用百货   18498       2025-09-25
4    53.0          华北     日用百货     -38       2024-01-07
..    ...         ...      ...     ...              ...
995  21.0          西南     生鲜食品    5123       2024-04-25
996  36.0          华南     日用百货   16784       2025-05-18
997  27.0          西南     生鲜食品    5610       2024-03-14
998  42.0          西北     日用百货    5198       2024-09-25
999  41.0          华东     生鲜食品    8101       2024-04-06

[1000 rows x 5 columns]


In [2]:
# --- 步骤 3: 检查数据初始状况 ---
if df is None:
    raise RuntimeError("没有提取到数据，请先运行上一个单元格并确认数据库连接成功。")

print("\n=== [初始状况] 数据基本信息 ===")
df.info()
print("\n=== [初始状况] 各列缺失值数量统计 ===")
print(df.isnull().sum())



=== [初始状况] 数据基本信息 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               869 non-null    float64
 1   region_name       935 non-null    object 
 2   category          950 non-null    object 
 3   amount            1000 non-null   int64  
 4   transaction_date  1000 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 39.2+ KB

=== [初始状况] 各列缺失值数量统计 ===
age                 131
region_name          65
category             50
amount                0
transaction_date      0
dtype: int64


In [3]:
# --- 步骤 4: 异常值与缺失值清洗 ---

# 4.1 异常金额：删除金额 <= 0 的交易（本数据中均为负值，推测为录入错误；
#     若实际是退款，应单独保留并标记，而不是直接删除）
n_raw = len(df)
df_clean = df[df["amount"] > 0].copy()
print(f"[4.1] 删除金额 <= 0 的记录：{n_raw - len(df_clean)} 条，剩余 {len(df_clean)} 条")


[4.1] 删除金额 <= 0 的记录：50 条，剩余 950 条


In [4]:
# 4.2 年龄缺失：中位数填充
# 中位数对极值不敏感，比均值更稳健。代价是缺失的记录会被统一成同一个值，压缩年龄分布的方差。
n_missing_age = int(df_clean["age"].isnull().sum())
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
print(f"[4.2] 年龄缺失 {n_missing_age} 条，已用中位数 {df_clean['age'].median()} 填充")


[4.2] 年龄缺失 125 条，已用中位数 38.0 填充


In [5]:
# 4.3 地区、类别缺失：直接删除
# region_name 缺失：部分用户的 region_id 本身为空（源数据缺失）。
# category 缺失：交易的 product_id 为空，或不在 products 表中（孤儿记录）。
# 这些信息无法由其他字段推断，用 Unknown 填充没有分析意义，因此直接删除。
n_before = len(df_clean)
df_clean = df_clean.dropna(subset=["region_name", "category"])
print(f"[4.3] 删除地区或类别缺失的记录：{n_before - len(df_clean)} 条，剩余 {len(df_clean)} 条")
print(f"      原始 {n_raw} 条 → 清洗后 {len(df_clean)} 条，保留比例 {len(df_clean) / n_raw:.1%}")


[4.3] 删除地区或类别缺失的记录：111 条，剩余 839 条
      原始 1000 条 → 清洗后 839 条，保留比例 83.9%


In [6]:
# --- 步骤 5: 检查清洗后数据的最终状况 ---
print("\n=== [清洗后状况] 各列缺失值数量统计 ===")
print(df_clean.isnull().sum())

print("\n=== [清洗后状况] 数据预览 ===")
print(df_clean.head())



=== [清洗后状况] 各列缺失值数量统计 ===
age                 0
region_name         0
category            0
amount              0
transaction_date    0
dtype: int64

=== [清洗后状况] 数据预览 ===
    age region_name category  amount transaction_date
0  53.0          华北     生鲜食品    8841       2024-09-14
1  31.0          华北     日用百货    1412       2025-04-26
2  45.0          华东     电子产品   16757       2025-08-05
3  35.0          华南     日用百货   18498       2025-09-25
5  23.0          华北     生鲜食品   25249       2024-08-17


In [7]:
# --- 附：缺失值来源核查（验证上面清洗决策的依据），完成后关闭连接 ---
if conn is not None and not conn.closed:
    checks = {
        "用户总数": "SELECT COUNT(*) FROM users",
        "age 为空的用户数": "SELECT COUNT(*) FROM users WHERE age IS NULL",
        "region_id 为空的用户数": "SELECT COUNT(*) FROM users WHERE region_id IS NULL",
        "product_id 为空的交易数": "SELECT COUNT(*) FROM transactions WHERE product_id IS NULL",
        "product_id 不在商品表中的交易数": """
            SELECT COUNT(*)
            FROM transactions t
            LEFT JOIN products p ON t.product_id = p.product_id
            WHERE t.product_id IS NOT NULL AND p.product_id IS NULL
        """,
    }
    with conn.cursor() as cur:
        for name, sql in checks.items():
            cur.execute(sql)
            print(f"{name}：{cur.fetchone()[0]}")
    conn.close()
    print("数据库连接已关闭。")
else:
    print("连接已关闭或不可用，请重新运行第一个单元格。")


用户总数：200
age 为空的用户数：24
region_id 为空的用户数：12
product_id 为空的交易数：30
product_id 不在商品表中的交易数：20
数据库连接已关闭。
